# Fine-tuning Phi-3-mini-4k-instruct com QLoRA no FinQA

**Objetivo:** Adaptar um LLM pequeno para o domínio de finanças corporativas usando QLoRA.

**Stack:** `transformers` · `peft` · `trl` · `bitsandbytes` · `datasets`

**Runtime:** T4 GPU (Runtime → Change runtime type → T4 GPU)

---

**Antes de rodar:** adicione seu `HF_TOKEN` nos Secrets do Colab (ícone 🔑 na barra lateral)

In [1]:
import torch

assert torch.cuda.is_available(), 'GPU nao detectada! Va em Runtime -> Change runtime type -> T4 GPU'

gpu = torch.cuda.get_device_properties(0)
vram_gb = gpu.total_memory / 1024**3
print(f'GPU: {gpu.name}')
print(f'VRAM: {vram_gb:.1f} GB')
print(f'CUDA: {torch.version.cuda}')

AssertionError: GPU nao detectada! Va em Runtime -> Change runtime type -> T4 GPU

## 1. Instalar dependencias

In [ ]:
%%capture
!pip install -q -U \
    transformers \
    peft \
    trl \
    bitsandbytes \
    datasets \
    accelerate \
    evaluate \
    rouge_score

print('Dependencias instaladas')

In [ ]:
import transformers, peft, trl, bitsandbytes, datasets
print(f"transformers: {transformers.__version__}")
print(f"peft: {peft.__version__}")
print(f"trl: {trl.__version__}")
print(f"bitsandbytes: {bitsandbytes.__version__}")
print(f"datasets: {datasets.__version__}")

transformers: 5.3.0
peft: 0.18.1
trl: 0.29.0
bitsandbytes: 0.49.2
datasets: 4.8.2


## 2. Configurar HF Token

In [ ]:
import os

# Se estiver no Colab web, usa Secrets. Se estiver no VS Code, pede via input.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    HF_TOKEN = input('Cole seu HF_TOKEN aqui: ').strip()

os.environ['HF_TOKEN'] = HF_TOKEN
assert HF_TOKEN, 'HF_TOKEN nao encontrado!'
print('HF_TOKEN carregado com sucesso')

HF_TOKEN carregado com sucesso


## 3. Carregar e preparar o FinQA

In [ ]:
import random
from datasets import load_dataset, Dataset

SYSTEM_PROMPT = (
    'You are an expert assistant in corporate finance. '
    'Answer questions about balance sheets, income statements, '
    'cash flow and accounting terms clearly and accurately.'
)

def format_phi3(example):
    question = example.get('question', '').strip()
    answer   = str(example.get('answer', '')).strip()
    context  = example.get('context', '').strip()

    if not question or not answer:
        return None

    if context:
        user_content = f'Financial context:\n{context}\n\nQuestion: {question}'
    else:
        user_content = f'Question: {question}'

    text = (
        f'<|system|>\n{SYSTEM_PROMPT}<|end|>\n'
        f'<|user|>\n{user_content}<|end|>\n'
        f'<|assistant|>\n{answer}<|end|>'
    )
    return {'text': text, 'question': question, 'answer': answer}

ds = load_dataset('virattt/financial-qa-10K')

formatted, skipped = [], 0
for ex in ds['train']:
    r = format_phi3(ex)
    if r:
        formatted.append(r)
    else:
        skipped += 1

print(f'Formatados: {len(formatted)} | Ignorados: {skipped}')

random.seed(42)
random.shuffle(formatted)
eval_size  = int(len(formatted) * 0.15)
train_data = formatted[eval_size:]
eval_data  = formatted[:eval_size]

train_ds = Dataset.from_list(train_data)
eval_ds  = Dataset.from_list(eval_data)

print(f'Treino: {len(train_ds)} | Eval: {len(eval_ds)}')
print('\n--- Exemplo ---')
print(train_ds[0]['text'][:500])

Formatados: 6997 | Ignorados: 3
Treino: 5948 | Eval: 1049

--- Exemplo ---
<|system|>
You are an expert assistant in corporate finance. Answer questions about balance sheets, income statements, cash flow and accounting terms clearly and accurately.<|end|>
<|user|>
Financial context:
In 2023, under pre-approved share repurchase programs, The Hershey Company repurchased shares valued at $27.4 million.

Question: What is the value of shares repurchased under the pre-approved program as stated in The Hershey Company's 2023 Form 10-K, for the year 2023?<|end|>
<|assistant|>


## 4. Carregar modelo base (baseline)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = 'microsoft/Phi-3-mini-4k-instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print('Carregando tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, token=HF_TOKEN, trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Carregando modelo em 4-bit...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    token=HF_TOKEN,
    attn_implementation='eager',
)

vram_used = torch.cuda.memory_allocated() / 1024**3
print(f'Modelo carregado! VRAM usada: {vram_used:.2f} GB')

Carregando tokenizer...
Carregando modelo em 4-bit...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Modelo carregado! VRAM usada: 2.11 GB


## 5. Avaliar baseline

In [ ]:
TEST_QUESTIONS = [
    'What is EBITDA and how is it calculated?',
    'What is the difference between gross profit and net profit?',
    'What does a current ratio below 1 indicate?',
    'How do you analyze the free cash flow of a company?',
    'What is the difference between accounts payable and accounts receivable?',
]

def generate_answer(model, tokenizer, question, max_new_tokens=200):
    prompt = (
        f'<|system|>\n{SYSTEM_PROMPT}<|end|>\n'
        f'<|user|>\nQuestion: {question}<|end|>\n'
        f'<|assistant|>\n'
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print('=== BASELINE - Phi-3-mini SEM fine-tuning ===')
baseline_answers = {}
for q in TEST_QUESTIONS:
    answer = generate_answer(base_model, tokenizer, q)
    baseline_answers[q] = answer
    print(f'\nQ: {q}')
    print(f'A: {answer[:300]}')
    print('-' * 50)

=== BASELINE - Phi-3-mini SEM fine-tuning ===

Q: What is EBITDA and how is it calculated?
A: EBITDA stands for Earnings Before Interest, Taxes, Depreciation, and Amortization. It is a measure used to evaluate a company'limpact of its operational performance without the effects of financing and accounting decisions. EBITDA is calculated by taking a company's net income and adding back intere
--------------------------------------------------

Q: What is the difference between gross profit and net profit?
A: Gross profit and net profit are two key financial metrics used to assess a company'ieves.


Gross profit is the profit a company makes after deducting the costs directly associated with the production of the goods it sells, which is known as the cost of goods sold (COGS). It is calculated by subtrac
--------------------------------------------------

Q: What does a current ratio below 1 indicate?
A: A current ratio below 1 indicates that a company may not have enough liquid assets t

## 6. Aplicar QLoRA e treinar

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

base_model = prepare_model_for_kbit_training(base_model)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

training_args = SFTConfig(
    output_dir='/content/checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=4,      # era 2
    per_device_eval_batch_size=4,       # era 2
    gradient_accumulation_steps=4,      # era 8 — effective batch continua 16
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=50,                    # trocou warmup_ratio
    weight_decay=0.01,
    optim='paged_adamw_8bit',
    bf16=True,
    max_length=256,                     # era 512 — maior impacto na velocidade
    dataset_text_field='text',
    packing=True,                       # era False — empacota sequências, muito mais rápido
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    logging_steps=50,
    report_to='none',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
)

train_ds_small = train_ds.select(range(1000))
eval_ds_small  = eval_ds.select(range(200))

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds_small,
    eval_dataset=eval_ds_small,
    args=training_args,
)

print('Iniciando treino...')
trainer.train()

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 8,912,896 || all params: 3,829,992,448 || trainable%: 0.2327


Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Packing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Iniciando treino...


Epoch,Training Loss,Validation Loss
1,No log,0.850831
2,1.366592,0.810862
3,0.849608,0.806769


TrainOutput(global_step=132, training_loss=1.0354177735068582, metrics={'train_runtime': 7440.7991, 'train_samples_per_second': 0.281, 'train_steps_per_second': 0.018, 'total_flos': 1.04353781349888e+16, 'train_loss': 1.0354177735068582})

In [ ]:
model.save_pretrained('/content/finance-phi3-qlora')

NameError: name 'model' is not defined

## 7. Salvar modelo

In [ ]:
SAVE_DIR = '/content/finance-phi3-qlora'
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'Modelo salvo em {SAVE_DIR}')
!ls -lh /content/finance-phi3-qlora/

NameError: name 'model' is not defined

## 8. Avaliar modelo fine-tunado

In [ ]:
print('=== FINE-TUNED - Phi-3-mini APOS fine-tuning ===')
finetuned_answers = {}
for q in TEST_QUESTIONS:
    answer = generate_answer(model, tokenizer, q)
    finetuned_answers[q] = answer
    print(f'\nQ: {q}')
    print(f'A: {answer[:300]}')
    print('-' * 50)

## 9. Comparacao ROUGE + tabela visual

In [ ]:
import evaluate
import pandas as pd
from IPython.display import display, HTML

rouge = evaluate.load('rouge')

reference_answers = {
    'What is EBITDA and how is it calculated?':
        'EBITDA stands for Earnings Before Interest, Taxes, Depreciation, and Amortization. It is calculated by adding back interest, taxes, depreciation and amortization to net income.',
    'What is the difference between gross profit and net profit?':
        'Gross profit is revenue minus cost of goods sold. Net profit is gross profit minus all operating expenses, interest, and taxes.',
    'What does a current ratio below 1 indicate?':
        'A current ratio below 1 indicates the company has more current liabilities than current assets, suggesting potential liquidity problems.',
    'How do you analyze the free cash flow of a company?':
        'Free cash flow is calculated as operating cash flow minus capital expenditures. Positive and growing FCF indicates financial health.',
    'What is the difference between accounts payable and accounts receivable?':
        'Accounts payable is money owed to suppliers. Accounts receivable is money owed by customers to the company.',
}

rows = []
for q in TEST_QUESTIONS:
    ref = reference_answers[q]
    base = baseline_answers[q]
    ft = finetuned_answers[q]
    r_base = rouge.compute(predictions=[base], references=[ref])
    r_ft = rouge.compute(predictions=[ft], references=[ref])
    rows.append({
        'Pergunta': q[:55] + '...',
        'ROUGE-L Base': round(r_base['rougeL'], 3),
        'ROUGE-L Fine-tuned': round(r_ft['rougeL'], 3),
        'Delta': round(r_ft['rougeL'] - r_base['rougeL'], 3),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print(f'\nMedia Base: {df["ROUGE-L Base"].mean():.3f}')
print(f'Media Fine-tuned: {df["ROUGE-L Fine-tuned"].mean():.3f}')
print(f'Melhoria media: +{df["Delta"].mean():.3f}')

html_rows = ''
for q in TEST_QUESTIONS:
    base = baseline_answers[q][:350]
    ft = finetuned_answers[q][:350]
    html_rows += f'<tr style="background:#f9f9f9"><td colspan="2" style="padding:10px;font-weight:bold">{q}</td></tr>'
    html_rows += f'<tr><td style="padding:10px;width:50%;border-right:1px solid #ddd;vertical-align:top"><b style="color:#c0392b">Base</b><br><br>{base}</td><td style="padding:10px;vertical-align:top"><b style="color:#27ae60">Fine-tuned</b><br><br>{ft}</td></tr>'
    html_rows += '<tr><td colspan="2" style="border-bottom:2px solid #eee"></td></tr>'

html = f'<table style="width:100%;border-collapse:collapse;font-size:13px"><thead><tr style="background:#2c3e50;color:white"><th style="padding:12px">Phi-3-mini Base</th><th style="padding:12px">Phi-3-mini Fine-tuned (FinQA)</th></tr></thead><tbody>{html_rows}</tbody></table>'
display(HTML(html))

## 10. Salvar CSV e fazer download

In [ ]:
df.to_csv('/content/results_comparison.csv', index=False)
print('Salvo em /content/results_comparison.csv')

try:
    from google.colab import files
    files.download('/content/results_comparison.csv')
except Exception:
    print('Download manual: clique no arquivo no painel Files do Colab')

## 11. (Opcional) Upload para o Hugging Face Hub

In [ ]:
# Descomente e preencha seu username para publicar
# HF_USERNAME = 'seu-username'
# REPO_NAME = 'phi3-mini-finance-qlora'
# model.push_to_hub(f'{HF_USERNAME}/{REPO_NAME}', token=HF_TOKEN)
# tokenizer.push_to_hub(f'{HF_USERNAME}/{REPO_NAME}', token=HF_TOKEN)
# print(f'Publicado em: https://huggingface.co/{HF_USERNAME}/{REPO_NAME}')